# BBOB Comparison Experiment Visualization

This notebook provides comprehensive visualization for BBOB comparison experiments run using `run-comparison-prompts-bbob.py`.

## Visualizations included:
1. **Convergence Analysis** - AOCC over evaluations
2. **Final Performance** - Box plots and statistical tables
3. **IOH Analysis** - ECDF and EAF plots (requires iohinspector)
4. **Per-Function Analysis** - Performance breakdown by BBOB function
5. **Code Evolution** - CEG graphs showing algorithm evolution
6. **Behavior Metrics** - Exploration vs exploitation analysis
7. **Cost Analysis** - Token usage comparison

In [ ]:
# Cell 1: Imports and Setup
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')

# Add parent directory to path for imports
sys.path.insert(0, os.path.abspath('..'))

# BLADE imports
from iohblade.loggers import ExperimentLogger
from iohblade.plots import (
    plot_convergence,
    plot_experiment_CEG,
    plot_boxplot_fitness,
    plot_boxplot_fitness_hue,
    fitness_table,
    plot_speedup,
    plot_token_usage,
)

# Optional imports
try:
    from iohblade.behaviour_metrics import compute_behavior_metrics
    HAS_BEHAVIOR_METRICS = True
except ImportError:
    HAS_BEHAVIOR_METRICS = False
    print('Warning: behaviour_metrics not available')

try:
    import iohinspector
    import polars as pl
    HAS_IOH_INSPECTOR = True
except ImportError:
    HAS_IOH_INSPECTOR = False
    print('Warning: iohinspector not available, ECDF/EAF plots will be skipped')

print('All imports successful.')
print(f'Behavior metrics available: {HAS_BEHAVIOR_METRICS}')
print(f'IOH Inspector available: {HAS_IOH_INSPECTOR}')

In [ ]:
# Cell 2: Load Experiment Data

# Configure the experiment directory here
# Option 1: Specify exact directory
# EXPERIMENT_DIR = '../results/COMPARISON-BBOB_20260216_120000'

# Option 2: Find most recent COMPARISON-BBOB experiment
results_dir = '../results'
bbob_dirs = [d for d in os.listdir(results_dir) if d.startswith('COMPARISON-BBOB_')]
if bbob_dirs:
    bbob_dirs.sort(reverse=True)  # Most recent first
    EXPERIMENT_DIR = os.path.join(results_dir, bbob_dirs[0])
    print(f'Using most recent experiment: {EXPERIMENT_DIR}')
else:
    raise FileNotFoundError('No COMPARISON-BBOB experiments found in results/')

# Load the experiment logger
logger = ExperimentLogger(EXPERIMENT_DIR, read=True)

# Get methods and problems
methods, problems = logger.get_methods_problems()
print(f'\nMethods found: {methods}')
print(f'Problems found: {problems}')

# Load experiment data
df = logger.get_data()
print(f'\nTotal runs: {len(df)}')
print(f'\nData columns: {df.columns.tolist()}')

# Display summary
df[['method_name', 'problem_name', 'seed']].head(10)

In [ ]:
# Cell 3: Convergence Plots
# Shows how AOCC (Area Over Convergence Curve) improves over algorithm evaluations

print('Generating convergence plots...')

# Get budget from progress file if available
progress_file = os.path.join(EXPERIMENT_DIR, 'progress.json')
if os.path.exists(progress_file):
    with open(progress_file) as f:
        progress = json.load(f)
    budget = progress.get('runs', [{}])[0].get('budget', 100)
else:
    budget = 100

print(f'Budget: {budget} evaluations')

# Plot convergence with mean and std
fig = plot_convergence(
    logger,
    metric='AOCC',
    aggregation='mean',
    variance_aggregation='std',
    budget=budget,
    save=False,
    return_fig=True,
    show_std=True,
)
plt.title('Convergence: Mean AOCC over Evaluations')
plt.tight_layout()
plt.savefig(os.path.join(EXPERIMENT_DIR, 'convergence.png'), dpi=150)
plt.show()

# Also plot with median for robustness
fig = plot_convergence(
    logger,
    metric='AOCC',
    aggregation='median',
    variance_aggregation='std',
    budget=budget,
    save=False,
    return_fig=True,
    show_std=True,
)
plt.title('Convergence: Median AOCC over Evaluations')
plt.tight_layout()
plt.savefig(os.path.join(EXPERIMENT_DIR, 'convergence_median.png'), dpi=150)
plt.show()

In [ ]:
# Cell 4: Box Plots of Final Fitness
# Shows distribution of final fitness scores across methods

print('Generating box plots...')

# Extract fitness from solution if not already in columns
df_plot = df.copy()
if 'fitness' not in df_plot.columns and 'solution' in df_plot.columns:
    df_plot['fitness'] = df_plot['solution'].apply(
        lambda sol: sol.get('fitness', float('nan')) if isinstance(sol, dict) else float('nan')
    )

# Create box plot
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(
    data=df_plot,
    x='method_name',
    y='fitness',
    ax=ax,
    palette='colorblind',
)
ax.set_xlabel('Method')
ax.set_ylabel('Final AOCC')
ax.set_title('Distribution of Final Fitness by Method')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(EXPERIMENT_DIR, 'boxplot_fitness.png'), dpi=150)
plt.show()

# Print summary statistics
print('\nSummary Statistics:')
summary = df_plot.groupby('method_name')['fitness'].agg(['mean', 'std', 'min', 'max', 'count'])
summary = summary.round(4)
print(summary)

In [ ]:
# Cell 5: Statistical Significance Table
# Generates a table with mean ± std and p-values for pairwise comparisons

print('Generating statistical significance table...')

try:
    table = fitness_table(logger, alpha=0.05, smaller_is_better=False)
    print('\nStatistical Significance Table (LaTeX format):')
    print(table.to_latex(escape=False))
    
    # Save as CSV for easy viewing
    table.to_csv(os.path.join(EXPERIMENT_DIR, 'significance_table.csv'))
    print(f'\nTable saved to: {EXPERIMENT_DIR}/significance_table.csv')
    
    # Display the table
    display(table)
except Exception as e:
    print(f'Error generating table: {e}')

In [ ]:
# Cell 6: IOH Analysis (ECDF/EAF Plots)
# Uses iohinspector for standard benchmarking visualizations

if HAS_IOH_INSPECTOR:
    print('Loading IOH data for ECDF/EAF analysis...')
    
    ioh_dir = os.path.join(EXPERIMENT_DIR, 'ioh')
    if os.path.exists(ioh_dir) and os.listdir(ioh_dir):
        try:
            # Load data using iohinspector
            manager = iohinspector.DataManager()
            manager.add_folder(ioh_dir)
            ioh_df = manager.load(include_data=True, include_stats=True)
            
            print(f'Loaded IOH data with {len(ioh_df)} records')
            
            # ECDF Plot
            print('\nGenerating ECDF plot...')
            fig = iohinspector.ecdf(
                ioh_df,
                x='evaluations',
                color='algorithm_name',
            )
            fig.savefig(os.path.join(EXPERIMENT_DIR, 'ecdf.png'), dpi=150)
            plt.show()
            
            # Per-function ECDF
            print('\nGenerating per-function ECDF...')
            fig = iohinspector.ecdf(
                ioh_df,
                x='evaluations',
                color='algorithm_name',
                facet_col='function_id',
                facet_col_wrap=5,
            )
            fig.savefig(os.path.join(EXPERIMENT_DIR, 'ecdf_per_function.png'), dpi=150)
            plt.show()
            
        except Exception as e:
            print(f'Error loading IOH data: {e}')
    else:
        print(f'No IOH data found in {ioh_dir}')
else:
    print('Skipping IOH analysis (iohinspector not installed)')
    print('Install with: pip install iohinspector')

In [ ]:
# Cell 7: Per-Function Performance Breakdown
# Analyzes performance on individual BBOB functions

print('Analyzing per-function performance...')

# Load detailed problem data
problem_data = logger.get_problem_data(problem_name='BBOB')

if not problem_data.empty:
    print(f'Loaded {len(problem_data)} evaluation records')
    
    # Check if performance_data is available in metadata
    if 'metadata' in problem_data.columns:
        # Extract per-function performance
        perf_records = []
        for _, row in problem_data.iterrows():
            metadata = row.get('metadata', {})
            if isinstance(metadata, dict) and 'performance_data' in metadata:
                for perf in metadata['performance_data']:
                    perf_records.append({
                        'method': row['method_name'],
                        'seed': row['seed'],
                        'eval_id': row['_id'],
                        'fid': perf['fid'],
                        'iid': perf['iid'],
                        'dim': perf['dim'],
                        'auc': perf['auc'],
                    })
        
        if perf_records:
            perf_df = pd.DataFrame(perf_records)
            
            # Aggregate by method and function
            func_perf = perf_df.groupby(['method', 'fid'])['auc'].agg(['mean', 'std']).reset_index()
            
            # Heatmap of mean performance
            pivot = func_perf.pivot(index='method', columns='fid', values='mean')
            
            fig, ax = plt.subplots(figsize=(14, 6))
            sns.heatmap(
                pivot,
                annot=True,
                fmt='.3f',
                cmap='RdYlGn',
                ax=ax,
                vmin=0,
                vmax=1,
            )
            ax.set_title('Mean AOCC by Method and Function ID')
            ax.set_xlabel('BBOB Function ID')
            ax.set_ylabel('Method')
            plt.tight_layout()
            plt.savefig(os.path.join(EXPERIMENT_DIR, 'heatmap_functions.png'), dpi=150)
            plt.show()
        else:
            print('No per-function performance data available in metadata')
    else:
        print('No metadata column found in problem data')
else:
    print('No problem data found')

In [ ]:
# Cell 8: Code Evolution Graphs (CEG)
# Visualizes how generated algorithms evolve over evaluations

print('Generating Code Evolution Graphs...')

try:
    # Plot CEG for all methods and runs
    plot_experiment_CEG(
        logger,
        metric='total_token_count',
        budget=budget,
        save=True,
        max_seeds=3,  # Limit for readability
    )
    print(f'CEG plots saved to {EXPERIMENT_DIR}')
except Exception as e:
    print(f'Error generating CEG: {e}')

In [ ]:
# Cell 9: Behavior Metrics Analysis
# Analyzes exploration vs exploitation behavior

if HAS_BEHAVIOR_METRICS:
    print('Computing behavior metrics...')
    
    try:
        # Load problem data
        problem_data = logger.get_problem_data(problem_name='BBOB')
        
        if not problem_data.empty:
            # Compute behavior metrics for each run
            behavior_records = []
            
            for (method, seed), group in problem_data.groupby(['method_name', 'seed']):
                group = group.sort_values('_id')
                try:
                    metrics = compute_behavior_metrics(group)
                    metrics['method'] = method
                    metrics['seed'] = seed
                    behavior_records.append(metrics)
                except Exception as e:
                    print(f'Error computing metrics for {method} seed {seed}: {e}')
            
            if behavior_records:
                behavior_df = pd.DataFrame(behavior_records)
                
                # Select key metrics to plot
                key_metrics = [
                    'avg_exploration_pct',
                    'success_rate',
                    'average_convergence_rate',
                    'longest_no_improvement_streak',
                ]
                
                available_metrics = [m for m in key_metrics if m in behavior_df.columns]
                
                if available_metrics:
                    # Box plots of behavior metrics
                    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
                    axes = axes.flatten()
                    
                    for i, metric in enumerate(available_metrics[:4]):
                        sns.boxplot(
                            data=behavior_df,
                            x='method',
                            y=metric,
                            ax=axes[i],
                            palette='colorblind',
                        )
                        axes[i].set_title(metric.replace('_', ' ').title())
                        axes[i].tick_params(axis='x', rotation=45)
                    
                    plt.tight_layout()
                    plt.savefig(os.path.join(EXPERIMENT_DIR, 'behavior_metrics.png'), dpi=150)
                    plt.show()
                    
                    # Summary table
                    print('\nBehavior Metrics Summary:')
                    summary = behavior_df.groupby('method')[available_metrics].mean().round(4)
                    display(summary)
                else:
                    print('No behavior metrics available')
            else:
                print('Could not compute behavior metrics')
        else:
            print('No problem data available')
    except Exception as e:
        print(f'Error in behavior analysis: {e}')
else:
    print('Skipping behavior metrics (module not available)')

In [ ]:
# Cell 10: Token Usage Analysis
# Compares computational cost across methods

print('Analyzing token usage...')

try:
    token_df = plot_token_usage(logger, save=False, return_df=True)
    
    if not token_df.empty:
        # Summary by method
        token_summary = token_df.groupby('method_name')['tokens'].agg(['sum', 'mean', 'std'])
        token_summary.columns = ['Total Tokens', 'Mean Tokens', 'Std Tokens']
        token_summary = token_summary.round(0).astype(int)
        
        print('\nToken Usage Summary:')
        display(token_summary)
        
        # Bar plot
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.barplot(
            data=token_df,
            x='method_name',
            y='tokens',
            estimator=sum,
            ax=ax,
            palette='colorblind',
        )
        ax.set_xlabel('Method')
        ax.set_ylabel('Total Tokens')
        ax.set_title('Token Usage by Method')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(os.path.join(EXPERIMENT_DIR, 'token_usage.png'), dpi=150)
        plt.show()
        
        # Efficiency: Performance per token
        df_efficiency = df.copy()
        if 'fitness' not in df_efficiency.columns:
            df_efficiency['fitness'] = df_efficiency['solution'].apply(
                lambda sol: sol.get('fitness', float('nan')) if isinstance(sol, dict) else float('nan')
            )
        
        # Merge with token data
        token_merged = token_df.groupby('method_name')['tokens'].sum().reset_index()
        fitness_merged = df_efficiency.groupby('method_name')['fitness'].mean().reset_index()
        efficiency = token_merged.merge(fitness_merged, on='method_name')
        efficiency['efficiency'] = efficiency['fitness'] / efficiency['tokens'] * 1e6  # Per million tokens
        
        print('\nEfficiency (Fitness per Million Tokens):')
        display(efficiency[['method_name', 'fitness', 'tokens', 'efficiency']].round(4))
    else:
        print('No token data available')
except Exception as e:
    print(f'Error analyzing tokens: {e}')

In [ ]:
# Cell 11: Summary Export
# Generates a comprehensive summary of the experiment

print('Generating experiment summary...')

summary = {
    'experiment_dir': EXPERIMENT_DIR,
    'methods': methods,
    'problems': problems,
    'total_runs': len(df),
    'budget': budget,
}

# Add performance summary
df_summary = df.copy()
if 'fitness' not in df_summary.columns:
    df_summary['fitness'] = df_summary['solution'].apply(
        lambda sol: sol.get('fitness', float('nan')) if isinstance(sol, dict) else float('nan')
    )

perf_summary = df_summary.groupby('method_name')['fitness'].agg(['mean', 'std', 'max']).to_dict()
summary['performance'] = perf_summary

# Save summary
summary_path = os.path.join(EXPERIMENT_DIR, 'experiment_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f'Summary saved to: {summary_path}')
print('\n' + '='*60)
print('EXPERIMENT SUMMARY')
print('='*60)
print(f'Directory: {EXPERIMENT_DIR}')
print(f'Methods: {methods}')
print(f'Total runs: {len(df)}')
print(f'Budget: {budget}')
print('\nPerformance (Mean ± Std):')
for method in methods:
    mean = perf_summary['mean'].get(method, 0)
    std = perf_summary['std'].get(method, 0)
    print(f'  {method}: {mean:.4f} ± {std:.4f}')
print('='*60)

In [ ]:
# Cell 12: Speed-up Analysis (Optional)
# Compare convergence speed between methods

if len(methods) >= 2:
    print('Generating speed-up plots...')
    
    # Compare GA-LLaMEA-Improved against each other method
    baseline = 'GA-LLaMEA-Improved' if 'GA-LLaMEA-Improved' in methods else methods[0]
    
    for method in methods:
        if method != baseline:
            try:
                fig = plot_speedup(
                    logger,
                    method_fast=baseline,
                    method_slow=method,
                    budget=budget,
                    save=False,
                    return_fig=True,
                )
                plt.title(f'Speed-up: {baseline} vs {method}')
                plt.tight_layout()
                plt.savefig(os.path.join(EXPERIMENT_DIR, f'speedup_{baseline}_vs_{method}.png'), dpi=150)
                plt.show()
            except Exception as e:
                print(f'Could not compute speedup for {method}: {e}')
else:
    print('Need at least 2 methods for speed-up analysis')